Statistical Engine
Build the core analysis logic first before touching any LLM:

1. Ingest a CSV with control/variant metrics
2. Calculate p-values, confidence intervals, relative lift
3. Flag underpowered tests (sample size checks)
4. Output a structured result dictionary

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

### 1. Ingest a CSV with control/variant metrics

In [2]:
df = pd.read_csv('../data/ab_test_experiments_v2.csv')
df.head(3)

,test_id,test_name,product_area,hypothesis,primary_metric,secondary_metric,device_type,audience_segment,start_date,end_date,...,relative_lift_pct,secondary_metric_control,secondary_metric_variant,avg_order_value,control_revenue,variant_revenue,revenue_delta,outcome_type,analyst,notes
0,EXP-1000,Category Filter UX v1,Acquisition,Clearer value proposition will improve activation,signup_rate,bounce_rate,Mobile,Returning Users,2023-10-15,2023-10-23,...,7.46,0.4066,0.3666,407.76,92969.28,54639.84,-38329.44,underpowered,Sam T.,Test targeting returning users on mobile. Hypo...
1,EXP-1001,Trust Badge Placement v2,Onboarding,Reducing friction will increase conversion,signup_rate,avg_session_duration,Desktop,Free Tier,2023-05-31,2023-07-12,...,14.69,234.1898,268.4530,431.28,5879640.24,2889144.72,-2990495.52,clear_winner,Sam T.,Test targeting free tier on desktop. Hypothesi...
2,EXP-1002,Progress Bar Addition v3,Onboarding,Urgency messaging will drive faster decisions,signup_rate,pages_per_session,Mobile,Premium Members,2023-04-23,2023-06-04,...,12.14,0.4045,0.4046,56.44,3447129.44,1656683.32,-1790446.12,clear_winner,Alex R.,Test targeting premium members on mobile. Hypo...


In [58]:
pre_test_data = df[['test_id', 'test_name', 'product_area', 'hypothesis', 'primary_metric',
       'secondary_metric', 'device_type', 'audience_segment', 'start_date',
       'end_date', 'test_duration_days', 'traffic_split_control_pct',
       'test_direction', 'test_tails', 'alpha', 'mde_relative_pct',
       'control_visitors', 'variant_visitors', 'total_visitors',
       'control_conversions', 'variant_conversions', 'relative_lift_pct',
       'secondary_metric_control', 'secondary_metric_variant',
       'avg_order_value', 'control_revenue', 'variant_revenue',
       'revenue_delta', 'analyst', 'notes']]

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   test_id                    200 non-null    object 
 1   test_name                  200 non-null    object 
 2   product_area               200 non-null    object 
 3   hypothesis                 200 non-null    object 
 4   primary_metric             200 non-null    object 
 5   secondary_metric           200 non-null    object 
 6   device_type                200 non-null    object 
 7   audience_segment           200 non-null    object 
 8   start_date                 200 non-null    object 
 9   end_date                   200 non-null    object 
 10  test_duration_days         200 non-null    int64  
 11  traffic_split_control_pct  200 non-null    int64  
 12  test_direction             200 non-null    object 
 13  test_tails                 200 non-null    object 

In [4]:
df.outcome_type.unique()

array(['underpowered', 'clear_winner', 'false_positive', 'clear_loser',
       'novelty_effect', 'neutral'], dtype=object)

### 2. Calculate p-values, confidence intervals, relative lift

In [18]:
def calculate_pvalue(control_visitors, variant_visitors, 
                     control_conversions, variant_conversions, 
                     test_tails, test_direction):
    # Conversion rates
    p_c = control_conversions / control_visitors
    p_v = variant_conversions / variant_visitors
 
    # Pooled proportion & standard error
    p_pool = (control_conversions + variant_conversions) / (control_visitors + variant_visitors)
    se = (p_pool * (1 - p_pool) * (1 / control_visitors + 1 / variant_visitors))**0.5
    
    # z-statistic (variant - control direction)
    z_stat = (p_v - p_c) / se
    # two tailed p-value calculation
    if test_tails == 'two_tailed':
        p_value = 2*(1-stats.norm.cdf(abs(z_stat)))
    else:
        if test_direction in ("superiority", "non_inferiority"):
            p_value = 1 - stats.norm.cdf(z_stat)   # P(Z > z_stat)
        else:
            p_value = stats.norm.cdf(z_stat)        # P(Z < z_stat)    
    
    return round(p_value, 4)

In [19]:
calculate_pvalue(1406, 769, 228, 134,  'one_tailed', 'superiority') #underpowered

0.2346

In [20]:
calculate_pvalue(215388,	92310,	13633,	6699,	'one_tailed',	'superiority') #winnter

0.0

In [41]:
def add_pvalues(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for i,row in df.iterrows():
        res = calculate_pvalue(
                control_conversions=int(row["control_conversions"]),
                control_visitors=int(row["control_visitors"]),
                variant_conversions=int(row["variant_conversions"]),
                variant_visitors=int(row["variant_visitors"]),
                test_tails=row["test_tails"],
                test_direction=row["test_direction"],
            )
        results.append(res)
    result_df = pd.DataFrame(results, columns = ['p_values'])
    out = df.copy()
    out['p_values'] = result_df['p_values']
    significance = (out['p_values'] < out['alpha'])*1
    out['Significant_Results']  = ['Significant' if x == 1 else 'Not-Significant' for x in significance]
    
    return out
    

In [48]:
test_df = add_pvalues(df)

In [49]:
test_df[['alpha', 'p_values', 'Significant_Results']].value_counts()

alpha  p_values  Significant_Results
0.05   0.0000    Significant            58
       1.0000    Not-Significant        12
0.01   0.0000    Significant            10
0.05   0.3253    Not-Significant         2
       0.6168    Not-Significant         2
                                        ..
       0.1019    Not-Significant         1
       0.0726    Not-Significant         1
       0.0679    Not-Significant         1
       0.0645    Not-Significant         1
       0.3405    Not-Significant         1
Length: 120, dtype: int64

### 3. Flag underpowered tests (sample size checks)

In [51]:
df.mde_relative_pct.unique()

array([ 3.6,  6.4,  5.2,  7.2,  4.3,  3.1, 12.3,  3.4,  5.7,  7.4,  3.9,
        6.1, 11. ,  2.8,  4.4,  6.3,  2.3,  9.3,  9.2, 13.2,  6. ,  3.5,
        8.4,  4.8,  5.8,  6.9,  7.5,  5.4, 14.3, 14.2,  2.1,  5.3,  9.8,
        4.2,  5.6,  6.7,  3.8,  9.7,  4.5,  9.9,  6.5,  7. ,  3.3,  8.9,
        2.9,  2.7,  4.9,  5.5,  3.7,  7.9,  8.8,  8.1,  5. , 12.2,  3. ,
        6.2,  7.7,  8.2,  6.6,  8.6, 13.7,  4.7,  3.2, 12.5,  9.6, 14.5,
        9.5,  8.5,  8.3,  9.1, 13.8,  4.1, 13.9, 13. ,  9. , 14.6, 13.1,
        2.6,  8. ,  9.4])

Calculate Required Sample Size for the original Test

The goal is to answer: how many visitors per variant do you need to reliably detect a meaningful difference in conversion rates?

The Inputs
Four values drive the calculation:

    Baseline rate (p₁) — the control group's conversion rate
    MDE — the smallest relative lift you care about detecting, e.g. 5% lift on a 10% baseline means p₂ = 0.105
    α (alpha) — your false positive tolerance (Type I error), typically 0.05
    β (beta) — your false negative tolerance (Type II error), typically 0.20, meaning power = 1 − β = 0.80
    
How the Tails Affect It

The only place tails enter the formula is in how you look up z_α:
Test Typez_α lookupTwo-tailedz at 1 − α/2 (e.g. α=0.05 → z = 1.96)One-tailedz at 1 − α (e.g. α=0.05 → z = 1.645)

In [62]:
df.columns
baseline_rate = df['control_conversion_rate']
MDE = df['mde_relative_pct']*.01
df['alpha']